In [ ]:
import numpy as np
from scipy.optimize import linprog
import math

class BBNodes:
    def __init__(self, c, A_ub, b_ub, bounds=None, name="0"):
        self.c = c
        self.A_ub = A_ub
        self.b_ub = b_ub
        self.bounds = bounds if bounds else [(0, None)] * len(c)
        self.name = name
        self.res = None

    def solve(self):
        # максимизация c*x эквивалентна минимизации -c*x
        self.res = linprog([-val for val in self.c], 
                           A_ub=self.A_ub, b_ub=self.b_ub, 
                           bounds=self.bounds, method='highs')
        return self.res.success

def solve_case(title, c, A_ub, b_ub):
    max_f = -float('inf')
    best_solutions = []
    
    print(f"\n--- {title.lower()} ---")
    # Формула целевой функции в текстовом виде
    formula = " + ".join([f"{c[i]}x_{i+1}" for i in range(len(c))])
    print(f"задача: f(x) = {formula} -> max")
    print("дерево поиска:")

    def branch(node, prefix="", connector="└── "):
        nonlocal max_f, best_solutions
        
        success = node.solve()
        node_label = f"узел-{node.name}"
        
        if not success:
            print(f"{prefix}{connector}{node_label}: недопустимо")
            return

        f_val = -node.res.fun
        x_val = node.res.x
        
        print(f"{prefix}{connector}{node_label}: f={f_val:.4f}, x={np.round(x_val, 3)}", end="")

        # Отсечение по границе
        if f_val <= max_f - 1e-7 and max_f != -float('inf'):
            print(" (отсечено: f <= f_best)")
            return

        # Поиск первой дробной переменной для ветвления
        idx_fractional = -1
        for i, val in enumerate(x_val):
            if not np.isclose(val, np.round(val), atol=1e-5):
                idx_fractional = i
                break
        
        # Если все переменные целые
        if idx_fractional == -1: 
            current_f = round(f_val, 4)
            if current_f > max_f + 1e-7:
                max_f = current_f
                best_solutions = [np.round(x_val).astype(int).tolist()]
                print(" (найден новый рекорд)")
            elif np.isclose(current_f, max_f, atol=1e-7):
                sol = np.round(x_val).astype(int).tolist()
                if sol not in best_solutions:
                    best_solutions.append(sol)
                print(" (альтернативный оптимальный план)")
            else:
                print(" (лист)")
            return

        # Процесс ветвления
        print(f" (ветвление по x{idx_fractional+1})")
        new_prefix = prefix + ("│   " if connector == "├── " else "    ")
        var_val = x_val[idx_fractional]
        
        # x_i <= floor(val)
        bounds_l = list(node.bounds)
        low, high = bounds_l[idx_fractional]
        bounds_l[idx_fractional] = (low, min(high if high is not None else float('inf'), math.floor(var_val + 1e-7)))
        branch(BBNodes(c, A_ub, b_ub, bounds_l, f"{node.name}.1"), new_prefix, "├── ")

        # x_i >= ceil(val)
        bounds_r = list(node.bounds)
        low, high = bounds_r[idx_fractional]
        bounds_r[idx_fractional] = (max(low, math.ceil(var_val - 1e-7)), high)
        branch(BBNodes(c, A_ub, b_ub, bounds_r, f"{node.name}.2"), new_prefix, "└── ")

    branch(BBNodes(c, A_ub, b_ub))
    
    print("-" * 60)
    if max_f == -float('inf'):
        print("результат: целочисленных решений в данной области не существует")
    else:
        print(f"результат: f* = {max_f}")
        for i, s in enumerate(best_solutions):
            print(f"  вариант {i+1}: x = {s}")

# solve_case("отсутствие решений", 
#            c=[3.0, 3.0], 
#            A_ub=[[3, 4], [4, -6], [-4, 3]], 
#            b_ub=[13, 1, -4])

# solve_case("единственное решение", 
#            c=[3.0, 3.0], 
#            A_ub=[[3, 4], [4, -6], [-4, 3]], 
#            b_ub=[13, 1, -1])

solve_case("несколько решений", 
           c=[3.0, 3.0], 
           A_ub=[[3, 4], [4, -6], [4, 3]], 
           b_ub=[13, 1, 10])


--- единственное решение ---
задача: f(x) = 3.0x_1 + 3.0x_2 -> max
дерево поиска:
└── узел-0: f=11.5588, x=[2.412 1.441] (ветвление по x1)
    ├── узел-0.1: f=11.2500, x=[2.   1.75] (ветвление по x2)
    │   ├── узел-0.1.1: f=8.2500, x=[1.75 1.  ] (ветвление по x1)
    │   │   ├── узел-0.1.1.1: f=6.0000, x=[1. 1.] (найден новый рекорд)
    │   │   └── узел-0.1.1.2: недопустимо
    │   └── узел-0.1.2: недопустимо
    └── узел-0.2: недопустимо
------------------------------------------------------------
результат: f* = 6.0
  вариант 1: x = [1, 1]
